# TransUNet + MaxSig — Stage-Placement Ablation

`TransUNet-MaxSig.ipynb` injects `MaxSigLayerNormalized` into the ResNet-50 hybrid backbone of
TransUNet, at the stages named `block1`, `block2`, `block3` (`resnet.body.block{1,2,3}` —
increasing depth / channel width, decreasing spatial resolution). The **default** configuration
trains with MaxSig at `("block1", "block2")`.

This notebook keeps that default run and adds the **two additional scenarios a reviewer is most
likely to ask for**, mirroring the same logic used in the U-Net ablation (isolate the shallowest
stage; test saturating every stage):

1. **Default (unchanged)** — MaxSig at `block1 + block2`
2. **MaxSig at `block1` only** — the shallowest / highest-resolution backbone stage.
   Reviewer framing: *"Did you test applying MaxSig only at the earliest ResNet stage and find it
   hurt performance?"* — directly analogous to the Stage-0 (64ch) question raised for the plain
   U-Net.
3. **MaxSig at all three stages (`block1 + block2 + block3`)** — adding it to the deepest stage too.
   Reviewer framing: *"Did you test adding MaxSig to every backbone stage, including the deepest
   one, to see whether the benefit compounds or performance degrades from over-regularizing the
   representation?"*

Each scenario is its own cell and reports only: **best validation Dice**, **best epoch**, and
**test Dice** — matching the reporting style used for the U-Net ablation.


In [1]:
"""
TransUNet + MaxSig — stage-placement ablation (imports & config)
"""

import os
from glob import glob
from time import perf_counter

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR

from TransUnet.vit_seg_modeling import VisionTransformer as ViT_seg
from TransUnet.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

ROOT = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/'
config = {
    "lr": 1e-4,
    "batch_size": 8,
    "epochs": 100,          # lower this (e.g. 40-60) for a faster 3-run sweep if needed
    "img_size": 256,
    "vit_name": "R50-ViT-B_16",
    "pretrained_path": "None",
}

RESULTS_DIR = ROOT + 'transunet_ablation_ckpts/'
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


/home/jupyter-r.yazdi/.local/lib/python3.10/site-packages/albumentations/check_version.py:49: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


## Dataset & dataloaders\n(unchanged from `TransUNet-MaxSig.ipynb`)

In [2]:
class mydataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms_img=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms_img = transforms_img
        self.mask_names = glob(masks_dir + "/*.tif") + glob(masks_dir + "/*.bmp") + glob(masks_dir + "/*.png")

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self, index):
        mask_name = self.mask_names[index]
        mask = cv2.imread(mask_name, cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)

        imgn = mask_name.replace('/masks/', '/images/')
        if imgn.endswith('.tif'):
            if not os.path.exists(imgn):
                imgn = imgn.replace('.tif', '.bmp')

        image = cv2.imread(imgn, cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)

        if self.transforms_img is not None:
            transformed = self.transforms_img(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        return image, mask


def build_dataloaders(root, batch_size, img_size=256):
    resize = A.Resize(img_size, img_size)

    train_transform = A.Compose([resize, ToTensorV2()])
    val_transform = A.Compose([resize, ToTensorV2()])
    test_transform = A.Compose([resize, ToTensorV2()])

    train_ds = mydataset(root + 'dataset/train/images/', root + 'dataset/train/masks/', train_transform)
    val_ds = mydataset(root + 'dataset/val/images/', root + 'dataset/val/masks/', val_transform)
    test_ds = mydataset(root + 'dataset/test/images/', root + 'dataset/test/masks/', test_transform)

    print("train:", len(train_ds), " val:", len(val_ds), " test:", len(test_ds))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_dataloaders(ROOT, config["batch_size"], config["img_size"])


train: 294  val: 63  test: 63


## MaxSig components & backbone injector\n(unchanged from `TransUNet-MaxSig.ipynb`)

In [2]:
class MaxSigLayerNormalized(nn.Module):
    def __init__(self, ch_in, ch_out, kernel_size, stride=1, use_sigmoid_input=True):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in, self.ch_out = ch_in, ch_out
        self.kernel_size, self.stride = kernel_size, stride
        self.use_sigmoid_input = use_sigmoid_input
        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))
        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))
        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x):
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"
        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(x_in, kernel_size=self.kernel_size,
                            padding=self.padding, stride=self.stride)
        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1
        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)
        b = torch.sigmoid(self.bias)
        D = torch.maximum(w, x_unfold) + b

        mean_stat = D.mean(dim=2)
        median_stat, _ = D.median(dim=2)
        center_values = x_unfold[:, :, self.center_index, :, :]

        out = (self.weight_center * center_values
               + self.weight_median * median_stat
               - mean_stat)
        return self.proj(out)


class GatedMaxSigWrapper(nn.Module):
    """gamma starts at 0 -> wrapped stage == original stage at init."""
    def __init__(self, inner_module, ch, kernel_sig=3, gamma_init=0.0, use_sigmoid_input=True):
        super().__init__()
        self.inner = inner_module
        self.maxsig = MaxSigLayerNormalized(ch, ch, kernel_sig, use_sigmoid_input=use_sigmoid_input)
        self.post = nn.Sequential(nn.BatchNorm2d(ch), nn.LeakyReLU(0.1, inplace=True))
        self.gamma = nn.Parameter(torch.tensor(gamma_init))

    def forward(self, x):
        x = self.inner(x)
        m = self.post(self.maxsig(x))
        return x + self.gamma * m


def inject_maxsig_into_resnet(vit_seg_model, stages=("block1", "block2"),
                              kernel_sig=3, gamma_init=0.0):
    resnet = vit_seg_model.transformer.embeddings.hybrid_model
    for stage_name in stages:
        stage_module = getattr(resnet.body, stage_name)
        last_unit = list(stage_module.children())[-1]
        out_ch = last_unit.gn3.num_channels if hasattr(last_unit, "gn3") \
            else last_unit.conv3.out_channels
        wrapped = GatedMaxSigWrapper(stage_module, ch=out_ch,
                                     kernel_sig=kernel_sig, gamma_init=gamma_init)
        setattr(resnet.body, stage_name, wrapped)
    return vit_seg_model


def build_model(vit_name, img_size, num_classes, pretrained_path,
                maxsig_stages=("block1", "block2"), maxsig_kernel=3):
    config_vit = CONFIGS_ViT_seg[vit_name]
    config_vit.n_classes = num_classes
    config_vit.n_skip = 3
    if "R50" in vit_name:
        config_vit.patches.grid = (img_size // 16, img_size // 16)

    model = ViT_seg(config_vit, img_size=img_size, num_classes=num_classes)
    if pretrained_path and os.path.exists(pretrained_path):
        model.load_from(weights=np.load(pretrained_path))
    else:
        print("WARNING: pretrained_path not found — training from random init.")

    if maxsig_stages:  # empty tuple => plain TransUNet, no MaxSig
        model = inject_maxsig_into_resnet(
            model, stages=maxsig_stages, kernel_sig=maxsig_kernel, gamma_init=0.0
        )
    return model


## Loss / metrics\n(unchanged from `TransUNet-MaxSig.ipynb`)

In [4]:
def dice_loss(probs, targets, eps=1e-7):
    inter = (probs * targets).sum(dim=(2, 3))
    union = (probs + targets).sum(dim=(2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()


def bce_dice_loss(logits, targets):
    probs = torch.sigmoid(logits)
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    return bce + dice_loss(probs, targets)


@torch.no_grad()
def dice_score(logits, targets, eps=1e-7):
    preds = (torch.sigmoid(logits) > 0.5).float()
    inter = (preds * targets).sum()
    union = (preds + targets).sum()
    return float((2 * inter + eps) / (union + eps))


def prep_batch(inputs, mask, device):
    x = inputs.to(device).float()
    if x.dim() == 3:
        x = x.unsqueeze(1)
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    y = mask.to(device).unsqueeze(1).float()
    return x, y


## Ablation helper

`run_scenario(run_name, maxsig_stages, ...)` builds a fresh TransUNet with MaxSig injected at the
given backbone stages (`()` = no MaxSig), trains it, evaluates test Dice with the best checkpoint,
and returns only `best_val_dice`, `best_epoch`, `test_dice` (per-epoch logs still print to track
progress, but the returned/reported summary is deliberately minimal).


In [5]:
def train_one(model, train_loader, val_loader, device, epochs, lr,
              loss_fn=bce_dice_loss, ckpt_path=None, verbose_every=10):
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = StepLR(optimizer, step_size=100, gamma=0.5)
    scaler = torch.amp.GradScaler('cuda') if device == "cuda" else None

    best_dice = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for inputs, mask in train_loader:
            x, y = prep_batch(inputs, mask, device)
            optimizer.zero_grad()
            if scaler is not None:
                with torch.autocast(device_type="cuda"):
                    logits = model(x)
                    loss = loss_fn(logits, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
        scheduler.step()

        model.eval()
        dice_running = 0.0
        with torch.no_grad():
            for inputs, mask in val_loader:
                x, y = prep_batch(inputs, mask, device)
                logits = model(x)
                dice_running += dice_score(logits, y)
        val_dice = dice_running / len(val_loader)

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch + 1
            if ckpt_path:
                torch.save(model.state_dict(), ckpt_path)

        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"epoch {epoch+1}/{epochs} | val_dice={val_dice:.4f} | best={best_dice:.4f}@{best_epoch}")

    return best_dice, best_epoch


def evaluate_test(model, test_loader, device, loss_fn, ckpt_path):
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path))
    model.to(device)
    model.eval()

    dice_running = 0.0
    with torch.no_grad():
        for inputs, mask in test_loader:
            x, y = prep_batch(inputs, mask, device)
            logits = model(x)
            dice_running += dice_score(logits, y)
    return dice_running / len(test_loader)


def run_scenario(run_name, maxsig_stages, epochs=config["epochs"], verbose_every=10):
    torch.manual_seed(42)

    model = build_model(
        vit_name=config["vit_name"], img_size=config["img_size"], num_classes=1,
        pretrained_path=config["pretrained_path"],
        maxsig_stages=maxsig_stages, maxsig_kernel=3,
    )

    ckpt_path = os.path.join(RESULTS_DIR, f"best-{run_name}.pt")

    best_val_dice, best_epoch = train_one(
        model, train_loader, val_loader, device,
        epochs=epochs, lr=config["lr"], loss_fn=bce_dice_loss,
        ckpt_path=ckpt_path, verbose_every=verbose_every,
    )

    test_dice = evaluate_test(model, test_loader, device, bce_dice_loss, ckpt_path)

    result = {
        "run_name": run_name,
        "maxsig_stages": maxsig_stages,
        "best_val_dice": best_val_dice,
        "best_epoch": best_epoch,
        "test_dice": test_dice,
    }
    print(f"[{run_name}] DONE | best_val_dice={best_val_dice:.4f} (epoch {best_epoch}) "
          f"| test_dice={test_dice:.4f}")
    return result


ablation_results = []


### Scenario 1 — Default: MaxSig at `block1 + block2`\n(the configuration already used in `TransUNet-MaxSig.ipynb`)

In [6]:
res = run_scenario(run_name="default_block1_block2", maxsig_stages=("block1", "block2"))
ablation_results.append(res)


/tmp/ipykernel_308100/1023665422.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


epoch 10/100 | val_dice=0.7114 | best=0.7114@10
epoch 20/100 | val_dice=0.7477 | best=0.7477@20
epoch 30/100 | val_dice=0.7447 | best=0.7543@22
epoch 40/100 | val_dice=0.7380 | best=0.7543@22
epoch 50/100 | val_dice=0.7346 | best=0.7543@22
epoch 60/100 | val_dice=0.7311 | best=0.7543@22
epoch 70/100 | val_dice=0.7162 | best=0.7543@22
epoch 80/100 | val_dice=0.6469 | best=0.7543@22
epoch 90/100 | val_dice=0.7522 | best=0.7543@22
epoch 100/100 | val_dice=0.7370 | best=0.7543@22
[default_block1_block2] DONE | best_val_dice=0.7543 (epoch 22) | test_dice=0.6887


### Scenario 2 — MaxSig at `block1` only (shallowest / highest-resolution backbone stage)
> Reviewer framing: *"Did you test applying MaxSig only at the earliest ResNet stage and find it
> hurt performance?"* — the TransUNet analogue of the Stage-0 (64ch) question raised for the plain
> U-Net.


In [7]:
res = run_scenario(run_name="block1_only", maxsig_stages=("block1",))
ablation_results.append(res)


/tmp/ipykernel_308100/1023665422.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


epoch 10/100 | val_dice=0.6055 | best=0.6965@8
epoch 20/100 | val_dice=0.5335 | best=0.7320@16
epoch 30/100 | val_dice=0.7393 | best=0.7448@21
epoch 40/100 | val_dice=0.7341 | best=0.7448@21
epoch 50/100 | val_dice=0.7333 | best=0.7448@21
epoch 60/100 | val_dice=0.7358 | best=0.7448@21
epoch 70/100 | val_dice=0.7286 | best=0.7448@21
epoch 80/100 | val_dice=0.4456 | best=0.7448@21
epoch 90/100 | val_dice=0.6530 | best=0.7448@21
epoch 100/100 | val_dice=0.7209 | best=0.7448@21
[block1_only] DONE | best_val_dice=0.7448 (epoch 21) | test_dice=0.7114


### Scenario 3 — MaxSig at all three backbone stages (`block1 + block2 + block3`)
> Reviewer framing: *"Did you test adding MaxSig to every backbone stage, including the deepest
> one, to see whether the benefit compounds or performance degrades from over-regularizing the
> representation?"*


In [8]:
res = run_scenario(run_name="all_blocks_1_2_3", maxsig_stages=("block1", "block2", "block3"))
ablation_results.append(res)


/tmp/ipykernel_308100/1023665422.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


epoch 10/100 | val_dice=0.6367 | best=0.6997@7
epoch 20/100 | val_dice=0.6433 | best=0.7309@12
epoch 30/100 | val_dice=0.7554 | best=0.7554@30
epoch 40/100 | val_dice=0.7429 | best=0.7589@33
epoch 50/100 | val_dice=0.7360 | best=0.7589@33
epoch 60/100 | val_dice=0.7487 | best=0.7589@33
epoch 70/100 | val_dice=0.7578 | best=0.7616@69
epoch 80/100 | val_dice=0.7164 | best=0.7616@69
epoch 90/100 | val_dice=0.7389 | best=0.7616@69
epoch 100/100 | val_dice=0.7409 | best=0.7616@69
[all_blocks_1_2_3] DONE | best_val_dice=0.7616 (epoch 69) | test_dice=0.7010


## Summary table

In [9]:
df = pd.DataFrame(ablation_results)
df = df[["run_name", "maxsig_stages", "best_val_dice", "best_epoch", "test_dice"]]
df


,run_name,maxsig_stages,best_val_dice,best_epoch,test_dice
0,default_block1_block2,"(block1, block2)",0.754316,22,0.688700
1,block1_only,"(block1,)",0.744786,21,0.711362
2,all_blocks_1_2_3,"(block1, block2, block3)",0.761552,69,0.701027


In [6]:

def compute_flops_fps(model, input_size=(1, 1, 512, 512), n_warmup=10, n_iters=50):
    """Reports GFLOPs (thop) and FPS (measured wall-clock throughput) for a model."""
    model.eval()
    dummy = torch.randn(*input_size).to(device)
    model=model.to(device)

    try:
        from thop import profile
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9
    except ImportError:
        print("`thop` not installed. Run `pip install thop` to get GFLOPs. Skipping GFLOPs.")
        gflops = None

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()
        for _ in range(n_iters):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = perf_counter()

    fps = n_iters / (t1 - t0)
    return gflops, fps

In [7]:
baseline_model = build_model(
        vit_name=config["vit_name"], img_size=config["img_size"], num_classes=1,
        pretrained_path=config["pretrained_path"],
        maxsig_stages=(), maxsig_kernel=3,
    )
gflops, fps = compute_flops_fps(baseline_model, input_size=(1, 3, 256, 256))
print(f"baseline | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


baseline | GFLOPs: 64.457 | FPS: 112.78


In [8]:
default_model = build_model(
        vit_name=config["vit_name"], img_size=config["img_size"], num_classes=1,
        pretrained_path=config["pretrained_path"],
        maxsig_stages=("block1", "block2"), maxsig_kernel=3,
    )
gflops, fps = compute_flops_fps(default_model, input_size=(1, 3, 256, 256))
print(f"default model | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


default model | GFLOPs: 65.527 | FPS: 69.74
